# Lesson 06 — Model Deployment

Covers: FastAPI model serving, ONNX export, Docker patterns, batch vs. real-time inference trade-offs.

## 1. FastAPI Model Server

In [ ]:
# Save this as serve.py and run with: uvicorn serve:app --reload
# pip install fastapi uvicorn pydantic

SERVE_PY = '''
import torch
import torch.nn as nn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from contextlib import asynccontextmanager
import numpy as np

# --- Model ---
class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 3))
    def forward(self, x):
        return self.net(x)

# --- Global model state ---
model = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global model
    model = SimpleClassifier()
    model.eval()
    # In production: model.load_state_dict(torch.load("model.pt"))
    print("Model loaded")
    yield
    print("Shutting down")

app = FastAPI(lifespan=lifespan)

class PredictRequest(BaseModel):
    features: list[float]

class PredictResponse(BaseModel):
    prediction: int
    probabilities: list[float]

@app.post("/predict", response_model=PredictResponse)
async def predict(req: PredictRequest):
    if len(req.features) != 10:
        raise HTTPException(400, "Expected 10 features")
    x = torch.tensor(req.features, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=-1).squeeze().tolist()
    return PredictResponse(prediction=int(np.argmax(probs)), probabilities=probs)

@app.get("/health")
async def health():
    return {"status": "ok"}
'''
print(SERVE_PY)


## 2. ONNX Export & Runtime Inference

In [ ]:
import torch
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 3))
    def forward(self, x):
        return self.fc(x)

model = SimpleNet()
model.eval()
dummy_input = torch.randn(1, 10)

# Export to ONNX
torch.onnx.export(
    model, dummy_input,
    "/tmp/model.onnx",
    input_names=["features"],
    output_names=["logits"],
    dynamic_axes={"features": {0: "batch_size"}, "logits": {0: "batch_size"}},
    opset_version=17,
)
print("ONNX model saved to /tmp/model.onnx")

# Inference with ONNX Runtime (pip install onnxruntime)
try:
    import onnxruntime as ort
    import numpy as np

    sess = ort.InferenceSession("/tmp/model.onnx", providers=["CPUExecutionProvider"])
    x_np = np.random.randn(4, 10).astype(np.float32)
    outputs = sess.run(None, {"features": x_np})
    print(f"ONNX inference output shape: {outputs[0].shape}")
except ImportError:
    print("Install onnxruntime: pip install onnxruntime")


## 3. Dockerfile for ML Serving

In [ ]:
DOCKERFILE = '''
# Multi-stage build — keeps final image lean
FROM python:3.11-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

FROM python:3.11-slim
WORKDIR /app
COPY --from=builder /install /usr/local
COPY . .

# Non-root user for security
RUN useradd -m appuser && chown -R appuser /app
USER appuser

EXPOSE 8000
CMD ["uvicorn", "serve:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''

print("Dockerfile:")
print(DOCKERFILE)

# requirements.txt should pin versions:
REQUIREMENTS = (
    "fastapi==0.111.0\n"
    "uvicorn[standard]==0.30.0\n"
    "torch==2.3.0+cpu\n"
    "onnxruntime==1.18.0\n"
    "pydantic==2.7.0\n"
    "numpy==1.26.4\n"
)
print("\nrequirements.txt:")
print(REQUIREMENTS)


## 4. Batch vs. Real-Time Inference Trade-offs

| | Real-Time (Online) | Batch (Offline) |
|---|---|---|
| Latency | <100ms target | Hours acceptable |
| Throughput | Lower (one at a time) | Higher (vectorised) |
| Infrastructure | FastAPI, Triton | Spark, Airflow, SageMaker Batch |
| GPU use | Wasteful (low utilisation) | Efficient (full batches) |
| Use case | User-facing APIs, fraud detection | Nightly scoring, dataset processing |

**Dynamic batching** (e.g. Triton Inference Server): buffer incoming requests for a few ms, group into a batch, return results. Best of both worlds for high-traffic services.

## 5. Model Versioning with MLflow

In [ ]:
# pip install mlflow

import mlflow
import mlflow.pytorch
import torch.nn as nn
import torch

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 3)
    def forward(self, x):
        return self.fc(x)

# Log a training run
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # local SQLite

with mlflow.start_run(run_name="baseline_v1"):
    model = MyModel()

    # Log hyperparameters
    mlflow.log_params({"lr": 1e-3, "epochs": 10, "batch_size": 64})

    # Log metrics (normally inside training loop)
    mlflow.log_metric("val_accuracy", 0.87, step=10)
    mlflow.log_metric("val_loss", 0.34, step=10)

    # Log model
    mlflow.pytorch.log_model(model, artifact_path="model")
    run_id = mlflow.active_run().info.run_id

print(f"Run ID: {run_id}")
print("View UI: mlflow ui --backend-store-uri sqlite:///mlflow.db")


## Interview Q&A

**Q: How would you serve 1000 req/s with a PyTorch model?**  
1. ONNX export + ONNX Runtime (removes Python overhead). 2. Triton Inference Server with dynamic batching. 3. Model quantisation (int8). 4. Horizontal scaling behind a load balancer. 5. Async FastAPI with multiple Uvicorn workers. Profile each bottleneck — it's usually memory bandwidth or Python GIL, not FLOPS.

**Q: Real-time vs batch — when does the line blur?**  
Micro-batching: buffer requests for 5-50ms to group into batches. Triton does this automatically. Threshold is when queuing delay < latency SLA benefit from batching.

**Q: Why ONNX over just saving a TorchScript?**  
ONNX is a cross-framework IR — you can export from PyTorch and run on TensorRT, CoreML, OpenVINO, or ONNX Runtime. TorchScript is PyTorch-only. ONNX also gets hardware-specific optimisations from each runtime.